# 01 — Summer Sample-Frame Audit

## Purpose

This notebook audits the sample construction used in the summer BIRLIUR hail pipeline.

The goal is not to redesign the original classifier. Instead, the goal is to determine whether its negative samples can be interpreted as **non-hail storms** for a storm-conditioned hail model.

Let

$$
S = \text{storm occurrence}, \qquad
H = \text{observed hail occurrence}.
$$

For storm-conditioned modeling, the desired negative class is

$$
S=1,\;H=0,
$$

that is, a storm is present but no hail is observed.

The key audit question is therefore:

> Does the existing summer label $y=0$ actually represent $S=1,H=0$, or does it represent a broader "no nearby hail report" condition?

A second audit checks whether the existing ERA5 block-construction design can be reused after the sample frame is changed.

## 1. Audit data

The summer pipeline generated a table of retained negative candidates around reported hail events.

Each retained row contains:

- the parent hail event;
- the candidate negative location;
- the candidate position identifier;
- whether another hail report was found nearby;
- the number of nearby hail reports;
- a unique negative-event identifier.

The large derived CSV is kept outside this repository. For local reproduction, it is expected at:

`../data/summer_pipeline/processed/valid_negative_records_all_years.csv`

The original summer source notebooks are also not redistributed in this repository. Their operational behavior is summarized later from the source-code audit.

In [1]:
from pathlib import Path

import pandas as pd


# ------------------------------------------------------------
# Locate repository data directory
#
# Supports running the notebook either from:
#   1. repo/notebooks/
#   2. repo root
# ------------------------------------------------------------

cwd = Path.cwd()

if (cwd / "data").exists():
    REPO_ROOT = cwd
elif (cwd.parent / "data").exists():
    REPO_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the repository data directory. "
        "Run this notebook from the repository root or notebooks/."
    )


NEG_PATH = (
    REPO_ROOT
    / "data"
    / "summer_pipeline"
    / "processed"
    / "valid_negative_records_all_years.csv"
)


if not NEG_PATH.exists():
    raise FileNotFoundError(
        "Summer negative-record table not found.\n"
        f"Expected local path: {NEG_PATH}\n\n"
        "See data/README.md for data-access instructions."
    )


# ------------------------------------------------------------
# Load retained summer negative candidates
# ------------------------------------------------------------

neg = pd.read_csv(
    NEG_PATH,
    parse_dates=["event_time"],
)


print(f"Rows: {len(neg):,}")
print(f"Columns: {neg.shape[1]}")
print(
    "Date range:",
    neg["event_time"].min(),
    "to",
    neg["event_time"].max(),
)

display(
    neg.head(3)
)

Rows: 1,516,901
Columns: 13
Date range: 1996-01-01 18:00:00 to 2025-05-31 17:02:00


,positive_EVENT_ID,event_time,positive_lat,positive_lon,negative_position_id,negative_center_lat,negative_center_lon,negative_has_hail,nearby_hail_count,nearby_event_ids,parent_event_id,sub_id,negative_event_id
0,5625665,1997-11-19 01:00:00,19.18,-155.9,0,20.18,-155.9,False,0,NaN,5625665,0,5625665_0
1,5625665,1997-11-19 01:00:00,19.18,-155.9,1,20.18,-154.9,False,0,NaN,5625665,1,5625665_1
2,5625665,1997-11-19 01:00:00,19.18,-155.9,2,19.18,-154.9,False,0,NaN,5625665,2,5625665_2


## 2. Operational meaning of the negative label

The first audit asks what conditions are actually guaranteed by the retained negative table.

Three invariants are especially important:

1. each retained negative has a unique negative-event identifier;
2. `negative_has_hail` is false for every retained row;
3. `nearby_hail_count` is zero for every retained row.

These checks establish what the summer negative label means operationally.

They do **not** establish whether a meteorological storm was present.

In [2]:
# ------------------------------------------------------------
# Core sample-frame audit
# ------------------------------------------------------------

audit_summary = pd.Series(
    {
        "rows": len(neg),
        "unique_negative_events": neg["negative_event_id"].nunique(),
        "unique_parent_hail_events": neg["parent_event_id"].nunique(),
        "duplicate_negative_ids": neg["negative_event_id"].duplicated().sum(),
        "rows_with_negative_has_hail_true": int(
            neg["negative_has_hail"].fillna(False).sum()
        ),
        "maximum_nearby_hail_count": int(
            neg["nearby_hail_count"].max()
        ),
        "first_event_time": neg["event_time"].min(),
        "last_event_time": neg["event_time"].max(),
    },
    name="value",
).to_frame()


display(audit_summary)


# ------------------------------------------------------------
# Assertions defining the retained negative table
# ------------------------------------------------------------

assert neg["negative_event_id"].is_unique
assert not neg["negative_has_hail"].fillna(False).any()
assert (neg["nearby_hail_count"] == 0).all()


print("All retained rows satisfy the report-free negative criteria.")

,value
rows,1516901
unique_negative_events,1516901
unique_parent_hail_events,229829
duplicate_negative_ids,0
rows_with_negative_has_hail_true,0
maximum_nearby_hail_count,0
first_event_time,1996-01-01 18:00:00
last_event_time,2025-05-31 17:02:00


All retained rows satisfy the report-free negative criteria.


In [3]:
# ------------------------------------------------------------
# Retention structure across candidate positions
# ------------------------------------------------------------

retained_by_position = (
    neg.groupby("sub_id")
    .size()
    .reindex(range(8), fill_value=0)
    .rename("retained_negative_count")
    .to_frame()
)

retained_by_position["fraction_of_retained_rows"] = (
    retained_by_position["retained_negative_count"]
    / len(neg)
)


# ------------------------------------------------------------
# Number of retained candidates per parent hail event
# ------------------------------------------------------------

retained_per_parent = (
    neg.groupby("parent_event_id")
    .size()
)

parent_retention_distribution = (
    retained_per_parent
    .value_counts()
    .sort_index()
    .rename_axis("retained_candidates")
    .rename("parent_event_count")
    .to_frame()
)


print("Retained negatives by candidate position:")
display(retained_by_position)

print("\nRetained candidate count per parent hail event:")
display(parent_retention_distribution)

Retained negatives by candidate position:


,retained_negative_count,fraction_of_retained_rows
sub_id,,
0,190534,0.125607
1,191288,0.126104
2,166716,0.109906
3,209577,0.138161
4,192006,0.126578
5,191699,0.126375
6,166366,0.109675
7,208715,0.137593



Retained candidate count per parent hail event:


,parent_event_count
retained_candidates,
1,488
2,1879
3,5429
4,12939
5,25756
6,44130
7,62612
8,76596


## 3. Sample-frame interpretation

Source-code review of the summer construction established the following procedure:

1. a reported NOAA hail event defines the parent event;
2. eight candidate negative centers are generated around that hail event;
3. a candidate is retained only when no hail report is found within ±1 hour and ±0.5° latitude/longitude of the candidate center.

Therefore, the operational meaning of the summer negative label is:

$$
y=0
\quad\Longrightarrow\quad
\text{no nearby observed NOAA hail report}.
$$

It does **not** imply

$$
y=0
\quad\Longrightarrow\quad
S=1,\;H=0.
$$

Without an independent storm-occurrence criterion, retained negatives can conceptually include both

$$
S=1,\;H=0
$$

and

$$
S=0,\;H=0.
$$

The existing negative table is also hail-centered: candidate locations are generated relative to reported hail events rather than from an independently identified population of storms.

### Main methodological implication

This is not necessarily a flaw for the original broad hail classifier.

It is, however, a **sample-frame mismatch** for the new conditional estimand

$$
P(H=1 \mid S=1, X),
$$

because that estimand requires the analysis population to contain independently identified storms before hail labels are assigned.

## 4. ERA5 block-construction audit

The summer positive and negative pipelines were also reviewed to determine which feature-extraction components could be reused.

| Component | Positive construction | Negative construction | Audit interpretation |
|---|---|---|---|
| Target-time alignment | Event time rounded to 1 hour | Event time rounded to 1 hour | Intended design is aligned |
| Temporal history | 48 hourly steps ending at target time | 48 hourly steps ending at target time | Intended design is aligned |
| Spatial patch | 5 × 5 ERA5 grid | 5 × 5 ERA5 grid | Intended design is aligned |
| Current feature schema | 83 channels in current source | 83 channels in current source | Current definitions are aligned |
| Historical executed output | A stale positive output shows 13 channels | Current source defines 83 channels | Historical HDF5 files require schema verification |
| Cross-month source handling | Per-time monthly sources are computed, but some positive instant/accum reads still reference the event-month source | Corresponding negative source uses the per-time source | Potential implementation asymmetry near month boundaries |

The important distinction is therefore:

> The **intended block design is largely symmetric**, but the audited source contains implementation/version caveats that prevent treating historical block files as automatically interchangeable.

In particular, a 48-hour history can cross a month boundary. The positive source computes a time-specific ERA5 source for each hour, but some feature reads still reference the original event-month source. That issue should be corrected or explicitly checked before historical positive blocks are reused across month boundaries.

This caveat is separate from the main sample-frame finding above.

## 5. Conclusion

This audit produces two separate findings.

### Sample-frame finding

The existing summer negative class is best interpreted as:

> a hail-centered candidate location with no nearby observed NOAA hail report.

It cannot be assumed to represent a non-hail storm because storm occurrence was not independently established before the hail label was assigned.

For the storm-conditioned target

$$
P(H=1 \mid S=1, X),
$$

the sample frame therefore needs to be rebuilt as:

$$
\text{radar}
\rightarrow
S
\rightarrow
\text{NOAA hail overlay}
\rightarrow
H.
$$

Storm occurrence must be defined independently of the hail reports that are later used to assign $H$.

### Feature-construction finding

The summer ERA5 block design provides useful reusable structure:

- hourly target-time alignment;
- a 48-hour history;
- a 5 × 5 spatial patch;
- a common current feature schema.

However, historical saved blocks should not be reused without checking feature-schema versioning and the positive-source month-boundary implementation.

### Next step

Notebook 02 develops and audits an MRMS radar-based proxy for storm occurrence.

The purpose is not to build a hail classifier from radar at this stage. It is to establish $S$ independently of $H$, so that hail and non-hail storms can subsequently be compared within a storm-first sample frame.